In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Concatenate, TimeDistributed, Flatten, RepeatVector, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical


In [2]:
import os
import json

def load_descriptions(filepath):
    """Подгружает описания картинок"""
    with open(filepath) as file:
        return json.load(file)
    
def load_images(image_dir):
    """Создаёт словарь с путями до картинок"""
    images = {}
    for img_name in os.listdir(image_dir):
        if img_name.find("json")==-1:
            images[img_name.split(".")[0]] = image_dir + img_name
    return images
    
images = load_images("./images/")
descriptions = load_descriptions("./images/descriptions.json")

In [3]:
def create_tokenizer(descriptions):
    """создаёт токенизатор"""
    lines = [desc for key in descriptions for desc in descriptions[key]]
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer

tokenizer = create_tokenizer(descriptions)

In [4]:
MAX_LENGTH = 32  # максимальное количество токенов на выходе
VOCAB_SIZE = len(tokenizer.word_index)+1 # количество токенов в токенизаторе

In [ ]:
# Загрузка и настройка ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
base_model.trainable = False

# Вход для изображений
image_input = Input(shape=(512, 512, 3))
image_features = base_model(image_input)
image_features = Dense(256, activation='relu')(image_features)
image_features = Reshape((256,))(image_features)

# Вход для текста
text_input = Input(shape=(1,))  # Ввод одного начального токена
embedded_text = Embedding(input_dim=VOCAB_SIZE, output_dim=256)(text_input)
embedded_text = Flatten()(embedded_text)  # Преобразуем в одномерный вектор

# Повторение признаков изображения и текста для каждого временного шага
image_features_repeated = RepeatVector(MAX_LENGTH)(image_features)
text_features_repeated = RepeatVector(MAX_LENGTH)(embedded_text)

# Объединение
decoder_input = Concatenate(axis=-1)([image_features_repeated, text_features_repeated])

# LSTM слой с возвращением последовательностей
lstm_out = LSTM(256, return_sequences=True)(decoder_input)

# Выходной слой для каждого временного шага
outputs = TimeDistributed(Dense(VOCAB_SIZE, activation='softmax'))(lstm_out)


model = Model(inputs=[image_input, text_input], outputs=outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

#model.summary()


In [ ]:
def create_sequences(tokenizer, desc_list, photo):
    """на основе словаря описаний и картинки организует данные для обучения"""
    seq = tokenizer.texts_to_sequences([desc_list[1:]])
    seq = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post')[0]
    input_image = photo
    input_seq = np.array([tokenizer.word_index['start']])
    output_seq = to_categorical(seq, num_classes=VOCAB_SIZE)
    return (input_image, input_seq), output_seq

def data_generator(descriptions, image_paths, tokenizer):
    def generator():
        for key, desc_list in descriptions.items():
            photo_path = image_paths[key]
            photo = load_img(photo_path, target_size=(512, 512))
            photo = img_to_array(photo)
            photo = preprocess_input(photo)
            yield create_sequences(tokenizer, desc_list, photo)

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            ((tf.TensorSpec(shape=(512, 512, 3), dtype=tf.float32),  # Изображение
            tf.TensorSpec(shape=(1,), dtype=tf.int32)),             # Начальный токен
            tf.TensorSpec(shape=(MAX_LENGTH, VOCAB_SIZE), dtype=tf.float32))  # Последовательность токенов в виде one-hot
        )
    )
    return dataset.repeat()

dataset = data_generator(descriptions, images, tokenizer)
dataset = dataset.batch(10).prefetch(tf.data.AUTOTUNE)

model.fit(dataset, steps_per_epoch=10, epochs=1)


In [5]:
model = tf.keras.models.load_model('models/15_09_2024.keras')

In [6]:
def find_image(dir, num):
    return next(filter(lambda f: f.startswith(f"{num}."), os.listdir(dir)))

def generate_desc(model, tokenizer, photo):
    sequence = [tokenizer.word_index['start']]

    sequence_input = np.array([sequence[-1]])
    result = np.argmax(model.predict([photo, sequence_input], verbose=0), axis=-1)
    return " ".join([tokenizer.index_word.get(token) for token in filter(None, result[0])])

In [ ]:
dir_path = "./images/"
for num in range(1, 152):
    # Загрузка изображения
    photo_path = dir_path+find_image(dir_path, num)
    photo = load_img(photo_path, target_size=(512, 512))
    photo = img_to_array(photo)
    photo = np.expand_dims(photo, axis=0)  # Добавляем размерность batch
    photo = preprocess_input(photo)

    description = generate_desc(model, tokenizer, photo)
    print(f"{num} ---", description)

In [ ]:
dir_path = "./train/"
photo_path = dir_path+find_image(dir_path, 20)
photo = load_img(photo_path, target_size=(512, 512))
photo = img_to_array(photo)
photo = np.expand_dims(photo, axis=0)  # Добавляем размерность batch
photo = preprocess_input(photo)

# Генерация описания
description = generate_desc(model, tokenizer, photo)
print(description)